In [ ]:
import os
import pandas as pd
import myfunctions
import total_annual_energy_consumption_dk as taec



In [ ]:
cwd = os.getcwd()
data_path = os.path.join(cwd, "data", "ProductionConsumptionSettlement_DK.csv")

In [ ]:
start_time = None
end_time = None
plot_type = "line"
aggregation = "daily"
title = "Gross Consumption MWh"

myfunctions.dataPlot(columns="GrossConsumptionMWh", file_path=data_path, start_time=start_time, end_time=end_time, ax=None, plot_type=plot_type, aggregation=aggregation, title=title)

### Combining total wind and solar production into single columns

In [ ]:
import csv

with open(data_path, "r", encoding="utf-8", newline="") as file:
    delimiter = csv.Sniffer().sniff(file.read(4096)).delimiter

data = pd.read_csv(
    data_path,
    sep=delimiter,
    decimal="," if delimiter == ";" else ".",
)

data["HourDK"] = pd.to_datetime(data["HourDK"])

wind_columns = [
    "OffshoreWindLt100MW_MWh",
    "OffshoreWindGe100MW_MWh",
    "OnshoreWindLt50kW_MWh",
    "OnshoreWindGe50kW_MWh",
]

solar_columns = [
    "SolarPowerLt10kW_MWh",
    "SolarPowerGe10Lt40kW_MWh",
    "SolarPowerGe40kW_MWh",
    "SolarPowerSelfConMWh"
]

data["TotalWindMWh"] = data[wind_columns].sum(axis=1)
data["TotalSolarMWh"] = data[solar_columns].sum(axis=1)

data.to_csv(data_path, sep=";", decimal=",", index=False)

In [ ]:
myfunctions.dataPlot(columns=["TotalWindMWh",
                              "TotalSolarMWh"],
                              file_path=data_path,
                              start_time=start_time,
                              end_time=end_time,
                              ax=None,
                              plot_type=plot_type,
                              aggregation="weekly",
                              title="Total Wind and Solar MWh")

### Calculating renewable energy deficits

In [ ]:
# no scaling
total_wind_production_pj = round(data["TotalWindMWh"].sum() * 3.6*10**(-6), 2)
total_solar_production_pj = round(data["TotalSolarMWh"].sum() * 3.6*10**(-6), 2)


print("Total annual consumption:", taec.TOTAL, "PJ")
print("Sum of total wind production:", total_wind_production_pj, "PJ")
print("Sum of total solar production:", total_solar_production_pj, "PJ")
print("Biomass contribution", taec.AVAILABLE_BIOMASS, "PJ")
print("Total renewable deficit:", round(taec.TOTAL - (total_wind_production_pj + total_solar_production_pj + taec.AVAILABLE_BIOMASS), 2), "PJ")

### Finding the optimal renewable_scaling_factor for deficit to be 0

In [ ]:
# Find the scaling factor that makes the aggregate renewable deficit zero.
available_variable_renewables_pj = total_wind_production_pj + total_solar_production_pj
required_renewable_energy_pj = taec.TOTAL - taec.AVAILABLE_BIOMASS

renewable_scaling_factor = required_renewable_energy_pj / available_variable_renewables_pj

zero_deficit = taec.TOTAL - taec.AVAILABLE_BIOMASS - renewable_scaling_factor * available_variable_renewables_pj

print(f"Optimal renewable scaling factor: {renewable_scaling_factor:.2f}")
print(f"Scaled wind production: {total_wind_production_pj * renewable_scaling_factor:.2f} PJ")
print(f"Scaled solar production: {total_solar_production_pj * renewable_scaling_factor:.2f} PJ")
print(f"Total renewable deficit: {zero_deficit:.12f} PJ")


#### Scale the electricity consumption time-series to match the cumulative direct electricity consumption (assumed to be the sum of households & industry) estimated in 2030

In [ ]:
total_gross_consumption_PJ = data['GrossConsumptionMWh'].sum() * 3.6 * 10**(-6)
direct_electricity_consumption_PJ = taec.HOUSEHOLDS + taec.INDUSTRY

electricity_consumption_scaling_factor = direct_electricity_consumption_PJ / total_gross_consumption_PJ

scaled_gross_consumption_PJ = total_gross_consumption_PJ * electricity_consumption_scaling_factor

print(f"Total gross electricity consumption in the grid: {total_gross_consumption_PJ:.2f} PJ")
print(f"Scaled gross electricity consumption in the grid: {scaled_gross_consumption_PJ:.2f} PJ")
print(f"Electricity consumption scaling factor: {electricity_consumption_scaling_factor:.2f}")

# Add a new column to the dataframe for the scaled gross consumption
data['ScaledGrossConsumptionMWh'] = data['GrossConsumptionMWh'] * electricity_consumption_scaling_factor

# Write the updated dataframe to file
data.to_csv(data_path, index=False)


#### Plot electricity production (wind + solar), direct electricity consumption (households & industry), and prosumption (production - consumption) time series. Provide the deduced scaling factors in the Caption of this Figure 1.

In [ ]:
myfunctions.dataPlot(columns=["TotalWindMWh",
                              "TotalSolarMWh",
                              "ScaledGrossConsumptionMWh"],
                              file_path=data_path,
                              start_time=start_time,
                              end_time=end_time,
                              ax=None,
                              plot_type=plot_type,
                              aggregation="weekly",
                              title="Total Wind and Solar MWh"
                              )